In [ ]:
from enum import Enum
from fractions import Fraction
from typing import Any
from music21 import clef, chord, harmony, key, metadata, meter, note, scale, stream, tempo, tie
from pydantic import BaseModel, computed_field, ConfigDict, Field


class AcceptableKeys(Enum):
    C_MAJOR = "C"
    A_MINOR = "Am"
    G_MAJOR = "G"
    E_MINOR = "Em"
    D_MAJOR = "D"
    B_MINOR = "Bm"
    A_MAJOR = "A"
    FS_MINOR = "F#m"
    E_MAJOR = "E"
    CS_MINOR = "C#m"
    B_MAJOR = "B"
    GS_MINOR = "G#m"
    FS_MAJOR = "F#"
    DS_MINOR = "D#m"
    CS_MAJOR = "C#"
    AS_MINOR = "A#m"
    F_MAJOR = "F"
    D_MINOR = "Dm"
    BF_MAJOR = "Bb"
    G_MINOR = "Gm"
    EF_MAJOR = "Eb"
    C_MINOR = "Cm"
    AF_MAJOR = "Ab"
    F_MINOR = "Fm"
    DF_MAJOR = "Db"
    BF_MINOR = "Bbm"
    GF_MAJOR = "Gb"
    EF_MINOR = "Ebm"
    CF_MAJOR = "Cb"
    AF_MINOR = "Abm"


class AcceptableTimeSigs(Enum):
    TWO_FOUR = "2/4"
    THREE_FOUR = "3/4"
    FOUR_FOUR = "4/4"
    FIVE_FOUR = "5/4"
    SIX_FOUR = "6/4"
    THREE_EIGHT = "3/8"
    FOUR_EIGHT = "4/8"
    FIVE_EIGHT = "5/8"
    SIX_EIGHT = "6/8"
    SEVEN_EIGHT = "7/8"
    NINE_EIGHT = "9/8"
    TWELVE_EIGHT = "12/8"
    TWO_TWO = "2/2"
    THREE_TWO = "3/2"


class BaseSong(BaseModel):
    model_config = ConfigDict(arbitrary_types_allowed=True)

    KEYS_MAP: dict[str, int] = {
        "C": 0,
        "Am": 0,
        "G": 1,
        "Em": 1,
        "D": 2,
        "Bm": 2,
        "A": 3,
        "F#m": 3,
        "E": 4,
        "C#m": 4,
        "B": 5,
        "G#m": 5,
        "F#": 6,
        "D#m": 6,
        "C#": 7,
        "A#m": 7,
        "F": -1,
        "Dm": -1,
        "Bb": -2,
        "Gm": -2,
        "Eb": -3,
        "Cm": -3,
        "Ab": -4,
        "Fm": -4,
        "Db": -5,
        "Bbm": -5,
        "Gb": -6,
        "Ebm": -6,
        "Cb": -7,
        "Abm": -7,
    }

    song_key_str: AcceptableKeys = Field(default_factory=lambda: AcceptableKeys.C_MAJOR)
    song_sig_str: AcceptableTimeSigs = Field(
        default_factory=lambda: AcceptableTimeSigs.FOUR_FOUR
    )

    song_key: key.KeySignature = key.KeySignature(0)
    song_sig: meter.TimeSignature = meter.TimeSignature("4/4")

    song: stream.Score = Field(default_factory=lambda: stream.Score())
    part: stream.Part = Field(default_factory=lambda: stream.Part())

    chords: dict[int, list[harmony.ChordSymbol]] = Field(
        default_factory=lambda: {
            0: [harmony.ChordSymbol("Dm7")],
            1: [harmony.ChordSymbol("G13")],
            2: [harmony.ChordSymbol("Cmaj7")],
            3: [harmony.ChordSymbol("A7")],
        }
    )
    
    def model_post_init(self, context: Any) -> None:
        measures: list[stream.Measure] = []

        for idx, ch in self.chords.items():
            m = stream.Measure()
            if idx == 0:
                m.append(self.song_key)
                m.append(self.song_sig)
            for c in ch:
                m.append(c)
            m.append(note.Note("C4", quarterLength=4))
            measures.append(m)
        
        for m in measures:
            self.part.append(m)
        
        self.song.append(self.part)
    
    def swingify(self, original_stream: stream.Score, swing_ratio=2) -> stream.Score:
        long_dur = Fraction(swing_ratio, swing_ratio + 1)
        short_dur = Fraction(1, swing_ratio + 1)

        for n in original_stream.recurse().notes:
            if n.quarterLength == 0.5:
                beat_pos = n.offset % 1.0

                if beat_pos == 0.0:
                    n.quarterLength = long_dur
                elif beat_pos == 0.5:
                    n.offset = float(int(n.offset)) + long_dur
                    n.quarterLength = short_dur
        
        return original_stream
    
    def show(self) -> None:
        self.song.show("musicxml.png")
    
    def play(self, swing: bool = True) -> None:
        swung_song: stream.Score = self.swingify(self.song)
        swung_song.show("midi")

In [ ]:
from music21 import clef, chord, harmony, key, metadata, meter, note, scale, stream, tempo, tie

song = stream.Score()
part = stream.Part()

s_key = key.KeySignature(-3)
s_sig = meter.TimeSignature("4/4")
s_tempo = tempo.MetronomeMark(66)

chords: dict[int, list[str]] = {
    0: ["Fm7", "B-7b9"],
    1: ["E-maj7", "A-maj7"],
    2: ["Dm7b5", "G7b9"],
    3: ["Cm7b5", "A7b9"],
    4: ["Dm7b5", "G7b9"],
    5: ["Cm7"],
    6: ["Am7b5", "D7b9"],
    7: ["Dm7b5", "G7"],
    8: ["Am7b5", "D7"],
    9: ["G7"],
    10: ["Gm7b5", "C7b9"],
    11: ["Fm7", "C7b9"],
    12: ["Fm7", "A-m7", "D-7"],
    13: ["G7#5", "C7b9"],
    14: ["Fm7", "B-7"],
    15: ["E-maj7", "G7b9", "C7b9"],
}

def _note(pitch: str, length: float, _tie: str = "None") -> note.Note:
    n = note.Note(pitch, quarterLength=length)
    if _tie != "None":
        n.tie = tie.Tie(_tie)
    return n

def _rest(length: float) -> note.Rest:
    return note.Rest(quarterLength=length)

notes: dict[int, list[note.Note | note.Rest]] = {
    0: [
        _rest(1),
        _note("G4", 0.333333333), _note("A-4", 0.333333333), _note("G4", 0.333333333),
        _note("G4", 0.75), _note("F4", 0.25),
        _note("G4", 0.5), _note("A-4", 0.5),
    ],
    1: [
        _note("B-4", 0.5), _note("B-4", 0.5, "start"),
        _note("B-4", 1, "stop"),
        _rest(1),
        _rest(0.333333333), _note("B-3", 0.333333333), _note("C4", 0.333333333),
    ],
    2: [
        _note("D4", 0.333333333), _note("E-4", 0.333333333), _note("D4", 0.333333333),
        _note("C#4", 2, "start"),
        _note("C#4", 0.25, "stop"), _note("D4", 0.25), _note("E-4", 0.25), _note("F4", 0.25),
    ],
    3: [
        _note("G4", 0.5), _note("G4", 0.5, "start"),
        _note("G4", 1, "stop"),
        _rest(2),
    ],
}

def construct_song(_song: stream.Score, _part: stream.Part, _key: key.KeySignature, _sig: meter.TimeSignature, _tempo: tempo.MetronomeMark, chords: dict[int, list[str]], notes: dict[int, list[note.Note | note.Rest]]):
    for idx in range(len(notes)):
        m = stream.Measure(number=idx + 1)

        if idx == 0:
            m.append(clef.TrebleClef())
            m.append(_key)
            m.append(_sig)
            m.append(_tempo)
        
        chord_list = chords.get(idx, [])
        if len(chord_list) == 2:
            m.insert(0.0, harmony.ChordSymbol(chord_list[0]))
            m.insert(2.0, harmony.ChordSymbol(chord_list[1]))
        elif len(chord_list) == 3:
            m.insert(0.0, harmony.ChordSymbol(chord_list[0]))
            m.insert(2.0, harmony.ChordSymbol(chord_list[1]))
            m.insert(3.0, harmony.ChordSymbol(chord_list[2]))
        else:
            m.insert(0.0, harmony.ChordSymbol(chord_list[0]))
        
        curr_offset = 0.0
        for el in notes[idx]:
            m.insert(curr_offset, el)
            curr_offset += el.quarterLength
        
        _part.append(m)
    
    _song.append(_part)
    return _song

song = construct_song(song, part, s_key, s_sig, s_tempo, chords, notes)

song.show("musicxml.png")
song.show("midi")